# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanaanwar25/flyrank-ml-internship-sana/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

REPO_DIR = "/content/flyrank-ml-internship-sana"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sanaanwar25/flyrank-ml-internship-sana.git

os.chdir(REPO_DIR)

print("Repository:", os.getcwd())
print("Repository exists:", os.path.exists(".git"))

Cloning into 'flyrank-ml-internship-sana'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 163 (delta 64), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 1.89 MiB | 13.95 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Repository: /content/flyrank-ml-internship-sana
Repository exists: True


In [2]:
# Install and import the packages used by the capstone
%pip -q install duckdb huggingface_hub scikit-learn pandas numpy

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score, f1_score

print("Packages loaded successfully.")

Packages loaded successfully.


In [3]:
# Read Hugging Face token from Colab Secret
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

print("Hugging Face token loaded:", bool(HF_TOKEN))

Hugging Face token loaded: True


In [4]:
# Connect to DuckDB
con = duckdb.connect()

print("DuckDB connection successful.")

DuckDB connection successful.


In [5]:
# Make sure DuckDB can access Hugging Face
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

print("DuckDB httpfs extension loaded.")

DuckDB httpfs extension loaded.


In [6]:
con.execute("DROP SECRET IF EXISTS hf_token")

con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [7]:
print(
    con.execute(
        "FROM duckdb_secrets()"
    ).fetchdf()[["name", "type"]]
)

       name         type
0  hf_token  huggingface


In [8]:
# Connect DuckDB to the FlyRank warehouse

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("FlyRank warehouse paths configured.")
print("Tables:", list(TABLES.keys()))

FlyRank warehouse paths configured.
Tables: ['dim_clients', 'dim_content', 'fact_daily', 'fact_query_90d']


In [9]:
test = con.execute(
    f"""
    SELECT *
    FROM {TABLES['dim_content']}
    LIMIT 5
    """
).fetchdf()

print("Rows loaded:", len(test))
display(test)

Rows loaded: 5


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


In [10]:
# Inspect the columns in the daily performance table

daily_columns = con.execute(
    f"""
    DESCRIBE SELECT *
    FROM {TABLES['fact_daily']}
    """
).fetchdf()

display(daily_columns)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [11]:
# Give DuckDB access to the Hugging Face token

con.execute("""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face authentication configured for DuckDB.")

Hugging Face authentication configured for DuckDB.


In [12]:
# Test Hugging Face access

print("Testing connection...")

test = con.execute("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
    LIMIT 5
""").fetchdf()

print("Rows loaded:", len(test))
display(test)

Testing connection...
Rows loaded: 5


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


# Abstract

This project investigates whether a machine-learning scoring approach can help prioritize pages for content review. The analysis uses anonymized search and content-performance data to identify pages that may deserve attention. A transparent baseline is compared with a learned model using a held-out evaluation design. The findings are observed and directional decision-support evidence, not causal proof or a prediction of Google's ranking algorithm.

## Introduction / Problem Statement

Content teams may have many pages to review, making it difficult to decide which pages deserve attention first. This project uses observed search-performance signals to identify content items associated with impression decline and provide a repeatable prioritization approach.

The decision supported by this analysis is which content items should receive earlier manual review. The output is decision-support only: it does not establish that a feature causes search performance changes and does not predict Google's ranking algorithm.

## 1. Question

*The research question and the decision it supports.*

This paper asks whether a leakage-safe ML model can rank content pages by their observed likelihood of needing attention better than a simple hand-written baseline rule. The decision supported is which pages a content team should review first. The goal is decision-support and prioritization, not causal proof or prediction of Google's ranking algorithm.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

I use the provided FlyRank full-release warehouse through DuckDB and Hugging Face. The analysis uses anonymized client, content, daily performance, and 90-day query-level data.

The feature-building workflow uses a 90-day historical window. Content items are retained when they have at least 100 impressions in the previous 30-day period, providing a minimum history threshold for the analysis.

The model uses pre-outcome search-performance and query-level signals, including previous-30-day impressions, visible query count, rare-query share, anonymous-query share, and top-query share. Client names, domains, URLs, private queries, credentials, and raw exports are not used in the public report.

Future outcome information is not used as a model feature in order to reduce leakage.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

The analysis defines a declining content item as one whose impressions in the last 30 days are less than 80% of its impressions in the previous 30 days.

The model uses five features available before the outcome window: previous-30-day impressions, visible query count, rare-query share, anonymous-query share, and top-query share.

A Random Forest classifier with 200 trees is used as the first learned model. The transparent baseline always predicts the majority class in the held-out test set.

Two validation designs are compared. First, a random 75/25 train-test split is used as a simple benchmark. Second, GroupShuffleSplit is used with client_hash_id as the grouping variable so that clients represented in the training data are separated from clients in the test data.

The grouped split is used to examine whether the observed signal survives a stricter cross-client evaluation. Label-derived and future-window information are excluded from the feature set to reduce leakage.

In [13]:
# Inspect the 90-day query table
query_columns = con.execute(
    f"""
    DESCRIBE SELECT *
    FROM {TABLES['fact_query_90d']}
    """
).fetchdf()

display(query_columns)

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [14]:
# Check the date range and basic size of the daily performance data

daily_summary = con.execute(
    f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(*) AS rows
    FROM {TABLES['fact_daily']}
    """
).fetchdf()

display(daily_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,rows
0,2025-01-27,2026-06-30,78835655


In [15]:
# Build the ML dataset
# Previous 30 days = features
# Following 30 days = outcome

features = con.execute(
    f"""
    WITH daily AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions
        FROM {TABLES['fact_daily']}
        GROUP BY 1, 2, 3
    ),

    content_history AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN report_date >= DATE '2026-04-01'
                    AND report_date < DATE '2026-05-01'
                    THEN impressions
                    ELSE 0
                END
            ) AS previous_30d_impressions,

            SUM(
                CASE
                    WHEN report_date >= DATE '2026-05-01'
                    AND report_date < DATE '2026-06-01'
                    THEN impressions
                    ELSE 0
                END
            ) AS last_30d_impressions

        FROM daily
        GROUP BY 1, 2
    )

    SELECT
        client_hash_id,
        content_hash_id,
        previous_30d_impressions,
        last_30d_impressions,

        CASE
            WHEN last_30d_impressions < 0.8 * previous_30d_impressions
            THEN 1
            ELSE 0
        END AS declining

    FROM content_history

    WHERE previous_30d_impressions >= 100
    """
).fetchdf()

print("ML rows:", len(features))
display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ML rows: 107144


,client_hash_id,content_hash_id,previous_30d_impressions,last_30d_impressions,declining
0,client_9958f0a7ae1df715,content_6afb4ede795f9338,237.0,291.0,0
1,client_9958f0a7ae1df715,content_f11131be425de35f,239.0,35.0,1
2,client_9958f0a7ae1df715,content_24d04ab3e5c628e0,208.0,221.0,0
3,client_9958f0a7ae1df715,content_e7b21ab2fbff3da9,105.0,109.0,0
4,client_9958f0a7ae1df715,content_848c35793ace1edf,142.0,256.0,0


In [16]:
# Show all query-table column names
print(query_columns["column_name"].tolist())

['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [17]:
# Find the columns we need for the 5 ML features

all_query_cols = query_columns["column_name"].tolist()

for col in all_query_cols:
    print(col)

client_hash_id
content_hash_id
query_hash_id
query_char_count
query_token_count
window_start
window_end
impressions_90d
clicks_90d
impressions_last30
clicks_last30
impressions_prev30
clicks_prev30
avg_position_90d
avg_position_last30
avg_position_prev30
content_total_impressions_90d
content_visible_query_count
rare_query_count
rare_impressions_share
anonymized_impressions_share


In [18]:
# Create the five model features from the 90-day query data

query_features = con.execute(
    f"""
    WITH latest_query AS (
        SELECT *
        FROM {TABLES['fact_query_90d']}
        WHERE window_end < DATE '2026-05-01'
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY client_hash_id, content_hash_id, query_hash_id
            ORDER BY window_end DESC
        ) = 1
    ),

    query_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,

            -- Feature 1: number of visible queries
            COUNT(DISTINCT query_hash_id) AS visible_query_count,

            -- Feature 2: average rare-query share
            AVG(rare_impressions_share) AS rare_query_share,

            -- Feature 3: average anonymous-query share
            AVG(anonymized_impressions_share) AS anonymous_query_share,

            -- Feature 4: share of impressions from the biggest query
            MAX(impressions_90d)
                / NULLIF(SUM(impressions_90d), 0)
                AS top_query_share

        FROM latest_query
        GROUP BY 1, 2
    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.previous_30d_impressions,
        f.last_30d_impressions,
        f.declining,

        COALESCE(q.visible_query_count, 0) AS visible_query_count,
        COALESCE(q.rare_query_share, 0) AS rare_query_share,
        COALESCE(q.anonymous_query_share, 0) AS anonymous_query_share,
        COALESCE(q.top_query_share, 0) AS top_query_share

    FROM features f
    LEFT JOIN query_agg q
        ON f.client_hash_id = q.client_hash_id
        AND f.content_hash_id = q.content_hash_id
    """
).fetchdf()

print("Final ML rows:", len(query_features))
print("Columns:")
print(query_features.columns.tolist())

display(query_features.head())

Final ML rows: 107144
Columns:
['client_hash_id', 'content_hash_id', 'previous_30d_impressions', 'last_30d_impressions', 'declining', 'visible_query_count', 'rare_query_share', 'anonymous_query_share', 'top_query_share']


,client_hash_id,content_hash_id,previous_30d_impressions,last_30d_impressions,declining,visible_query_count,rare_query_share,anonymous_query_share,top_query_share
0,client_62f4a7e64f5e0096,content_fea6cdd65e3c0ba7,643.0,146.0,1,0,0.0,0.0,0.0
1,client_62f4a7e64f5e0096,content_9a4fbcb07fe897a6,138.0,207.0,0,0,0.0,0.0,0.0
2,client_20259bd6705d81d4,content_8ae3050a4c913371,856.0,273.0,1,0,0.0,0.0,0.0
3,client_20259bd6705d81d4,content_2d642167cb59f587,2341.0,645.0,1,0,0.0,0.0,0.0
4,client_20259bd6705d81d4,content_e60213409e8c351a,6184.0,947.0,1,0,0.0,0.0,0.0


In [19]:
# Train and evaluate the Random Forest on a random 75/25 split

FEATURE_COLS = [
    "previous_30d_impressions",
    "visible_query_count",
    "rare_query_share",
    "anonymous_query_share",
    "top_query_share",
]

X = query_features[FEATURE_COLS].fillna(0)
y = query_features["declining"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print("Random split results")
print("--------------------")
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("F1 score:", round(f1_score(y_test, pred), 3))

print("\nClassification report:")
print(classification_report(y_test, pred))

Random split results
--------------------
Train rows: 80358
Test rows: 26786
Accuracy: 0.518
F1 score: 0.585

Classification report:
              precision    recall  f1-score   support

           0       0.47      0.39      0.43     12316
           1       0.55      0.63      0.59     14470

    accuracy                           0.52     26786
   macro avg       0.51      0.51      0.51     26786
weighted avg       0.51      0.52      0.51     26786



In [20]:
# Compare against a majority-class baseline
# and evaluate generalization across clients

# Majority-class baseline on the same random test set
majority_class = y_train.mode()[0]
baseline_pred = np.full(len(y_test), majority_class)

print("RANDOM SPLIT — MODEL VS BASELINE")
print("--------------------------------")
print("Model accuracy:", round(accuracy_score(y_test, pred), 3))
print("Model F1:", round(f1_score(y_test, pred), 3))
print("Baseline accuracy:", round(accuracy_score(y_test, baseline_pred), 3))
print("Baseline F1:", round(f1_score(y_test, baseline_pred), 3))
print("Majority class:", majority_class)

# Client-grouped train/test split
groups = query_features["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_group_train = X.iloc[train_idx]
X_group_test = X.iloc[test_idx]
y_group_train = y.iloc[train_idx]
y_group_test = y.iloc[test_idx]

rf_group = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_group.fit(X_group_train, y_group_train)

group_pred = rf_group.predict(X_group_test)

group_majority = y_group_train.mode()[0]
group_baseline_pred = np.full(
    len(y_group_test),
    group_majority
)

print("\nCLIENT-GROUPED SPLIT")
print("--------------------")
print("Train rows:", len(X_group_train))
print("Test rows:", len(X_group_test))
print("Test clients:", query_features.iloc[test_idx]["client_hash_id"].nunique())

print("\nModel accuracy:",
      round(accuracy_score(y_group_test, group_pred), 3))

print("Model macro F1:",
      round(f1_score(y_group_test, group_pred, average="macro"), 3))

print("Model weighted F1:",
      round(f1_score(y_group_test, group_pred, average="weighted"), 3))

print("Baseline accuracy:",
      round(accuracy_score(y_group_test, group_baseline_pred), 3))

print("\nClassification report:")
print(classification_report(y_group_test, group_pred))

RANDOM SPLIT — MODEL VS BASELINE
--------------------------------
Model accuracy: 0.518
Model F1: 0.585
Baseline accuracy: 0.54
Baseline F1: 0.701
Majority class: 1

CLIENT-GROUPED SPLIT
--------------------
Train rows: 102004
Test rows: 5140
Test clients: 12

Model accuracy: 0.474
Model macro F1: 0.471
Model weighted F1: 0.463
Baseline accuracy: 0.392

Classification report:
              precision    recall  f1-score   support

           0       0.63      0.33      0.43      3123
           1       0.40      0.70      0.51      2017

    accuracy                           0.47      5140
   macro avg       0.51      0.51      0.47      5140
weighted avg       0.54      0.47      0.46      5140



## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The 90-day feature workflow produced 107,144 content items with enough history for analysis. The decline label marks an item as declining when impressions in the last 30 days are less than 80% of impressions in the previous 30 days.

On the random 75/25 split, the Random Forest achieved 0.518 accuracy and 0.587 F1-score. The majority-class baseline achieved 0.540 accuracy and 0.701 F1-score. Therefore, the model did not outperform the baseline on this split.

Using a client-grouped holdout, the test set contained 5,140 rows from 12 clients. The model achieved 0.468 accuracy, 0.464 macro F1, and 0.454 weighted F1. The baseline accuracy was 0.392.

These results are observational and directional. In this run, the learned model did not clearly outperform the simple baseline, so the model should not be presented as a superior predictor. The workflow is still useful as a reproducible decision-support experiment.

## 5. Limitations

*What this work cannot claim.*

This analysis cannot prove that any feature causes a change in search performance. It cannot explain or predict Google's ranking algorithm, and it cannot guarantee that a recommended page will improve after editing. The results are limited to the observed anonymized dataset, its time windows, its label definition, and the validation design. The ranked queue should therefore be treated as directional decision-support for review prioritization.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*



The model is intended to support content-review prioritization rather than make automatic publishing decisions.

A practical workflow is to review higher-risk content first and combine the model's result with editorial judgment. A high model score indicates that an item resembles content associated with the observed decline label in this dataset; it does not guarantee that a refresh will improve performance.

Reviewers should check the underlying content and search-performance context before taking action. Items should not be automatically rewritten, removed, or published based only on the model result.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*



The main analysis artifact is the capstone notebook containing the feature construction, model training, random holdout evaluation, and client-grouped evaluation.

The notebook records the assumptions, feature definitions, label definition, validation design, and measured results so that the analysis can be inspected and reproduced from the repository.

## Reproducibility

The analysis is contained in this capstone notebook and the supporting assignment notebooks in the repository. The workflow documents the data preparation, feature construction, model training, validation design, and measured results.

The analysis is designed to be reproducible from the provided anonymized dataset without exposing client names, domains, private queries, credentials, or raw exports.

## 8. Acknowledgments & Data Credit

This project was completed as part of the FlyRank ML internship workflow. The analysis uses the provided anonymized, public-safe dataset and follows the project guidance for responsible use of the data.

Data and project context are credited to FlyRank.

[FlyRank](https://flyrank.ai)

## 5-Minute Demo Outline

**0:00–0:30 — Problem:** Explain the goal of identifying content items associated with observed impression decline.

**0:30–1:30 — Data:** Explain the anonymized FlyRank data, the 90-day historical window, and the minimum previous-30-day impression threshold.

**1:30–2:30 — Method:** Explain the decline label, five model features, Random Forest model, majority-class baseline, and leakage controls.

**2:30–3:30 — Results:** Show the random-split results and then the stricter client-grouped results.

**3:30–4:30 — Generalization:** Explain why the grouped client split is useful for checking whether the signal survives across clients.

**4:30–5:00 — Limitations:** Explain that the results are directional decision-support evidence, not causal proof or a prediction of Google's ranking algorithm.

## Social-Post Cut

Built a leakage-aware ML workflow to identify content items associated with observed impression decline using anonymized search-performance data. I compared a Random Forest model with a transparent majority-class baseline and then tested the model with a client-grouped holdout to examine cross-client generalization. The results are intended as practical decision-support evidence, not causal proof or a prediction of Google's ranking algorithm.

## Employer-Facing Summary

I built a leakage-aware ML workflow using anonymized search-performance data to identify content items associated with observed impression decline. I evaluated a Random Forest model against a transparent majority-class baseline and then used a client-grouped holdout to examine whether the signal generalized across clients. The project demonstrates an end-to-end approach to feature engineering, leakage-aware validation, and honest communication of model limitations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
